# 深度解析: `Gate`水工建筑物

**相关模块:** `preissmann_model/structures.py`

## 目的
本教程旨在深入解析`Gate`（闸门）这一水工建筑物的物理原理和在模型中的实现方式。与之前仅展示如何使用`Gate`的示例不同，本教程将聚焦于其内部的工作机制，并验证其行为是否符合水力学理论。

此笔记本将：
1.  解释`Gate`模型所基于的物理方程——**孔口出流公式（Orifice Equation）**。
2.  详细说明`Gate`类的每一个参数的物理意义。
3.  通过一系列数值实验，验证模型计算出的流量是否与理论公式一致。
4.  绘制不同工况下（不同水位差、不同开度）的“流量-水头”关系曲线。

## 1. 物理原理: 孔口出流公式

模型中的`Gate`模拟的是一个**淹没式堰孔（Submerged Orifice）**的流动。其流量可以通过经典的孔口出流公式来计算：

$$ Q = C_d \cdot A_g \cdot \sqrt{2g \Delta H} $$

其中：
- `Q`: 通过闸门的流量 (m³/s)
- `C_d`: 流量系数（Discharge Coefficient），一个无量纲参数，反映了能量损失，通常在0.6左右。
- `A_g`: 闸门过水断面的面积 (m²), 等于 `opening_height` × `width`。
- `g`: 重力加速度 (9.81 m/s²)。
- `ΔH`: 闸门上、下游的水头差 (m), 即 `Z_up - Z_down`。

在Preissmann求解器中，由于需要求解线性方程组，这个非线性公式被**线性化**为一个关于`dZ_up`, `dZ_down`, `dQ`的形式，其系数由`get_linearized_equation`方法计算。

## 2. 环境设置和参数定义

我们先定义一个`Gate`实例所需的参数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.structures import Gate

# --- 闸门参数 ---
# opening_height: 闸门开启高度 (m)
# width: 闸门宽度 (m)
# C_d: 流量系数
gate_params = {
    'name': 'test_gate',
    'node_index': 1, # 在这个独立的测试中，节点索引不重要
    'opening_height': 1.0, 
    'width': 10.0,
    'C_d': 0.6
}

gate = Gate(**gate_params)
print("Gate对象创建成功。")

## 3. 数值实验与理论验证

为了验证`Gate`类的实现是否正确，我们可以直接调用其内部的流量计算逻辑，并与我们自己根据理论公式计算的结果进行比较。注意，`Gate`类本身没有直接计算流量的方法，流量是在求解器中作为未知数求解的。但我们可以反过来，给定`Q`和`Z_down`，求解`Z_up`。

从`get_linearized_equation`的`rhs`项 `-(Q - C*sqrt(Z_up - Z_down))` 可知，当系统收敛时，`rhs`应趋近于0，即 `Q = C*sqrt(Z_up - Z_down)`。我们可以利用这个关系进行验证。

In [ ]:
def calculate_theoretical_q(gate, head_diff):
    if head_diff <= 0:
        return 0
    C = gate.C_d * (gate.opening_height * gate.width) * np.sqrt(2 * 9.81)
    return C * np.sqrt(head_diff)

# --- 实验: 改变水头差，观察流量变化 ---
head_diffs = np.linspace(0.1, 5.0, 20)
q_theoretical = [calculate_theoretical_q(gate, h) for h in head_diffs]

plt.figure(figsize=(8, 6))
plt.plot(head_diffs, q_theoretical, 'b-o', label='Theoretical Discharge')
plt.title(f'Gate Rating Curve (Opening={gate.opening_height}m)')
plt.xlabel('Head Difference (Z_up - Z_down) [m]')
plt.ylabel('Discharge (Q) [m³/s]')
plt.grid(True)
plt.legend()
plt.show()

## 4. 不同开度下的流量曲线

我们还可以进一步探索闸门开度对泄流能力的影响。我们为几种不同的开度绘制流量-水头关系曲线。

In [ ]:
openings = [0.5, 1.0, 1.5, 2.0]
head_diffs = np.linspace(0.1, 5.0, 20)

plt.figure(figsize=(10, 7))

for opening in openings:
    temp_gate = Gate(name='temp', node_index=1, opening_height=opening, width=10, C_d=0.6)
    q_values = [calculate_theoretical_q(temp_gate, h) for h in head_diffs]
    plt.plot(head_diffs, q_values, '-o', label=f'Opening = {opening} m')

plt.title('Impact of Gate Opening on Rating Curve')
plt.xlabel('Head Difference (Z_up - Z_down) [m]')
plt.ylabel('Discharge (Q) [m³/s]')
plt.legend()
plt.grid(True)
plt.show()

### 结论

从上图中可以清晰地看到：
- **流量随水头差增加**: 对于给定的开度，上下游的水头差越大，通过闸门的流量就越大，这符合物理直觉。
- **流量随开度增加**: 对于给定的水头差，闸门的开度越大，过水面积`Ag`越大，因此流量也越大。

这些数值实验验证了`Gate`类的实现与标准的水力学理论是一致的，从而增强了我们对模型模拟结果的信心。